# Visão Computacional com CNNs e Transformers
## 🎓 Faculdade Infnet — Pós-Graduação
### 🎯 Laboratório Prático: Segmentação Semântica com U-Net e DeepLabV3 (Predição Densa, Máscaras por Pixel e Fine-Tuning no PyTorch)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/allanspadini/curso-vision-transformers-infnet/blob/main/aula_01_cnn_architectures/aula_01_semantic_segmentation.ipynb)

---

### 📖 Contextualização Pedagógica: O que é Segmentação Semântica?

Nas tarefas anteriores de visão computacional:
- **Classificação**: Prevê uma única classe para a imagem inteira ($y \in \{1, \dots, C\}$).
- **Detecção de Objetos (Faster R-CNN / YOLO)**: Localiza caixas delimitadoras retangulares aproximadas $[x_1, y_1, x_2, y_2]$ para cada instância.

No entanto, em aplicações de alta criticidade como **medicina diagnóstica** (delimitação precisa de tumores e lesões em ressonâncias), **veículos autônomos** (distinção exata entre calçada, asfalto, faixa de pedestre e vegetação) e **sensoriamento remoto / satélites**, aproximações retangulares são insuficientes.

A **Segmentação Semântica (*Semantic Segmentation*)** resolve a tarefa de **Predição Densa (*Dense Pixel-Level Prediction*)**, onde a rede classifica **CADA PIXEL INDIVIDUAL** $p_{i,j}$ da imagem em uma categoria semântica específica, gerando uma **máscara matricial 2D** de saída de mesma resolução espacial $[H, W]$.

---

### 🎯 Objetivos de Aprendizagem
1. **Compreender a Predição Densa**: Entender como tensores de imagem $[3, H, W]$ são mapeados para matrizes de probabilidade $[C, H, W]$ e convertidos em máscaras de classe discretas $[H, W]$.
2. **Inferência Zero-Shot com Modelo de Estado da Arte (DeepLabV3)**: Utilizar o modelo pré-treinado no COCO/Pascal VOC (`torchvision.models.segmentation.deeplabv3_resnet50`) diretamente em imagens reais do mundo real.
3. **Implementar a Arquitetura U-Net do Zero em PyTorch**: Construir o caminho de contração (*Encoder*), o gargalo (*Bottleneck*), as conexões de salto (*Skip Connections*) e o caminho de expansão (*Decoder* com `ConvTranspose2d`).
4. **Download de Dataset Real via `kagglehub`**: Baixar o dataset urbano **Cityscapes Image Pairs** (`dansbecker/cityscapes-image-pairs`) e implementar o pipeline de carregamento com DataLoaders.
5. **Funções de Custo Especializadas e Métricas**: Aplicar **Cross-Entropy por Pixel** combinada com **Dice Loss** e monitorar a métrica canônica **mIoU (*Mean Intersection over Union*)**.
6. **Motor de Renderização Visual e Alpha-Blending**: Criar sobreposições translúcidas (*Color Overlays*) de alta qualidade para visualização de resultados clínicos e industriais.


## 1. Configuração do Ambiente e Verificação de GPU

Vamos instalar o `kagglehub` e importar os módulos fundamentais do PyTorch e TorchVision.


In [ ]:
# Instalar kagglehub para download ultrarrápido
!pip install -q kagglehub

import os
import sys
import time
import math
import random
import urllib.request
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchvision
import torchvision.transforms as transforms
from torchvision.models.segmentation import deeplabv3_resnet50, DeepLabV3_ResNet50_Weights

# Fixar sementes para reprodutibilidade
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

# Dispositivo de execução
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🔥 Dispositivo de Execução: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM Total: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("   ⚠️ Atenção: Nenhuma GPU detectada. No Colab, vá em: Ambiente de Execução -> Alterar tipo de ambiente -> T4 GPU")


## 2. Parte 1 — Inferência Zero-Shot com DeepLabV3 Pré-Treinado no MS-COCO

O **DeepLabV3 (Chen et al., 2017)** é uma das arquiteturas mais consagradas para segmentação semântica, utilizando **Convoluções Atros (*Atrous / Dilated Convolutions*)** e o módulo **ASPP (*Atrous Spatial Pyramid Pooling*)** para capturar contexto em múltiplas escalas sem perder resolução espacial.

Vamos carregar o modelo pré-treinado do TorchVision e executar inferência direta em uma imagem real.


In [ ]:
# 1. Carregar DeepLabV3 com backbone ResNet-50 pré-treinado no COCO (21 classes Pascal VOC)
weights = DeepLabV3_ResNet50_Weights.DEFAULT
deeplab_model = deeplabv3_resnet50(weights=weights)
deeplab_model.to(device)
deeplab_model.eval()

# Transformações oficiais do modelo (Normalização ImageNet)
preprocess = weights.transforms()

# 21 Classes do Pascal VOC / COCO
VOC_CLASSES = [
    'background', 'aeroplane', 'bicycle', 'bird', 'boat', 'bottle', 'bus',
    'car', 'cat', 'dog', 'horse', 'motorbike', 'person', 'pottedplant',
    'sheep', 'sofa', 'train', 'tvmonitor', 'chair', 'cow', 'diningtable'
]

print("✅ Modelo DeepLabV3 ResNet-50 carregado com sucesso!")
print(f"   • Total de Classes Pré-treinadas: {len(VOC_CLASSES)}")


### 2.1 Baixando Imagem de Teste do Mundo Real

Vamos baixar uma imagem urbana com pessoas e veículos para inspecionar a saída do modelo.


In [ ]:
os.makedirs("sample_images", exist_ok=True)
sample_img_path = os.path.join("sample_images", "street_scene.jpg")

# URL de imagem de cena urbana pública
IMG_URL = "https://upload.wikimedia.org/wikipedia/commons/thumb/d/d3/City_traffic_in_New_York_City.jpg/800px-City_traffic_in_New_York_City.jpg"
if not os.path.exists(sample_img_path):
    urllib.request.urlretrieve(IMG_URL, sample_img_path)
    print(f"📥 Imagem de teste baixada em: {sample_img_path}")
else:
    print(f"✅ Imagem pronta em: {sample_img_path}")

input_image = Image.open(sample_img_path).convert("RGB")
print(f"📸 Resolução Original da Imagem: {input_image.size[0]} x {input_image.size[1]} pixels")


### 2.2 Desmistificando as Saídas do Modelo de Segmentação

Ao passar um tensor de imagem $[1, 3, H, W]$ pelo DeepLabV3:
1. A rede retorna um dicionário onde a chave `'out'` contém o tensor de **Logits** de formato:
   $$\text{Tensor Logits} \in \mathbb{R}^{\text{Batch} \times \text{Num\_Classes} \times H \times W}$$
2. Para cada pixel $(i, j)$, temos um vetor de $C = 21$ valores reais representando a pontuação não-normalizada de cada classe.
3. Aplicamos a função **`torch.argmax(logits, dim=1)`** ao longo do eixo das classes para colapsar o tensor tridimensional em uma **matriz 2D de índices de classes discretos** $[H, W]$:
   $$\text{Máscara}_{i,j} = \arg\max_{c} \left( \text{Logits}_{c, i, j} \right)$$


In [ ]:
# Aplicar pré-processamento e mover para GPU
input_tensor = preprocess(input_image).unsqueeze(0).to(device)

print(f"📥 Formato do Tensor de Entrada:  {input_tensor.shape} (Batch, Canais, Altura, Largura)")

with torch.no_grad():
    output = deeplab_model(input_tensor)['out']

print(f"📤 Formato do Tensor de Saída:    {output.shape} (Batch, 21 Classes, Altura, Largura)")

# Aplicar Softmax para obter probabilidades [0.0, 1.0]
probabilities = F.softmax(output, dim=1)[0]

# Obter a classe dominante para cada pixel via argmax
predicted_mask = torch.argmax(output, dim=1)[0].cpu().numpy()

print(f"🎭 Formato da Máscara 2D Final:   {predicted_mask.shape} (Altura x Largura de inteiros)")
unique_classes = np.unique(predicted_mask)
print(f"🏷️ Classes Detectadas na Imagem:  {[VOC_CLASSES[c] for c in unique_classes]}")


### 2.3 Renderização Visual com Alpha-Blend Overlay

Vamos implementar a visualização com um mapa de cores vibrante (*Colormap*) sobreposto à imagem original com transparência ($lpha = 0.55$).


In [ ]:
# Tabela de Cores Pascal VOC (21 cores distintas)
VOC_COLORMAP = np.array([
    [0, 0, 0],       # 0: Background (Preto)
    [128, 0, 0],     # 1: Aeroplane
    [0, 128, 0],     # 2: Bicycle
    [128, 128, 0],   # 3: Bird
    [0, 0, 128],     # 4: Boat
    [128, 0, 128],   # 5: Bottle
    [0, 128, 128],   # 6: Bus (Cyan)
    [128, 128, 128], # 7: Car (Cinza)
    [64, 0, 0],      # 8: Cat
    [192, 0, 0],     # 9: Dog
    [64, 128, 0],    # 10: Horse
    [192, 128, 0],   # 11: Motorbike
    [64, 0, 128],    # 12: Person (Roxo)
    [192, 0, 128],   # 13: Potted Plant
    [64, 128, 128],  # 14: Sheep
    [192, 128, 128], # 15: Sofa
    [0, 64, 0],      # 16: Train
    [128, 64, 0],    # 17: Tv/Monitor
    [0, 192, 0],     # 18: Chair
    [128, 192, 0],   # 19: Cow
    [0, 64, 128]     # 20: Dining Table
], dtype=np.uint8)

def decode_segmap(mask, colormap=VOC_COLORMAP):
    """Converte máscara de IDs [H, W] para imagem RGB colorida [H, W, 3]"""
    rgb_mask = colormap[mask]
    return rgb_mask

# Redimensionar imagem original para o tamanho da máscara prevista
img_resized = input_image.resize((predicted_mask.shape[1], predicted_mask.shape[0]))
img_np = np.array(img_resized)
mask_colored = decode_segmap(predicted_mask)

# Alpha-blending (Mescla translúcida)
alpha = 0.55
overlay = (img_np * (1 - alpha) + mask_colored * alpha).astype(np.uint8)

# Plotagem em 3 colunas
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(img_np)
axes[0].set_title("1. Imagem Original RGB", fontsize=12, fontweight='bold', color='#0A345D')
axes[0].axis('off')

axes[1].imshow(mask_colored)
axes[1].set_title("2. Máscara Semântica Isolada (DeepLabV3)", fontsize=12, fontweight='bold', color='#0A345D')
axes[1].axis('off')

axes[2].imshow(overlay)
axes[2].set_title("3. Alpha-Blend Overlay (Máscara + Foto)", fontsize=12, fontweight='bold', color='#0A345D')
axes[2].axis('off')

plt.suptitle("🎯 Segmentação Semântica Zero-Shot com DeepLabV3 ResNet-50", fontsize=14, fontweight='bold', color='#0A345D')
plt.tight_layout()
plt.show()


## 3. Parte 2 — Download do Dataset de Segmentação via `kagglehub`

Para treinar nossa própria rede do zero, utilizaremos o dataset **Cityscapes Image Pairs** (`dansbecker/cityscapes-image-pairs`), contendo pares de imagens urbanas e suas respectivas máscaras semânticas com ruas, calçadas, prédios, carros e vegetação.


In [ ]:
import kagglehub

print("📥 Baixando dataset Cityscapes via kagglehub...")
start_time = time.time()

cityscapes_root = kagglehub.dataset_download("dansbecker/cityscapes-image-pairs")

print(f"✅ Download concluído em {time.time() - start_time:.2f}s!")
print(f"📂 Diretório Local: {cityscapes_root}")

train_folder = os.path.join(cityscapes_root, "cityscapes_data", "train")
val_folder = os.path.join(cityscapes_root, "cityscapes_data", "val")

# Se a pasta não estiver aninhada:
if not os.path.exists(train_folder):
    train_folder = os.path.join(cityscapes_root, "train")
    val_folder = os.path.join(cityscapes_root, "val")

print(f"📊 Amostras de Treino: {len(os.listdir(train_folder))}")
print(f"📊 Amostras de Validação: {len(os.listdir(val_folder))}")


## 4. Implementação do `Dataset` para Pares Imagem-Máscara

Cada arquivo `.jpg` do Cityscapes Image Pairs possui dimensão $512 \times 256$, onde:
- A metade esquerda ($256 \times 256$) é a **Foto RGB**.
- A metade direita ($256 \times 256$) é a **Máscara de Segmentação**.

Vamos construir o `Dataset` PyTorch que separa as metades, normaliza a imagem e quantiza as cores da máscara em $C = 4$ superclasses urbanas:
1. `0: Fundo / Construções / Céu`
2. `1: Vias (Asfalto / Calçadas)`
3. `2: Veículos (Carros / Ônibus)`
4. `3: Vegetação (Árvores / Grama)`


In [ ]:
class CityscapesPairsDataset(Dataset):
    def __init__(self, folder_path, img_size=(128, 128)):
        self.folder_path = folder_path
        self.img_files = sorted(os.listdir(folder_path))
        self.img_size = img_size
        
        self.transform_img = transforms.Compose([
            transforms.Resize(img_size),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        
    def __len__(self):
        return len(self.img_files)
        
    def __getitem__(self, idx):
        file_path = os.path.join(self.folder_path, self.img_files[idx])
        combined = Image.open(file_path).convert("RGB")
        
        w, h = combined.size
        # Dividir imagem combinada em 2 metades (Esquerda: Foto, Direita: Máscara)
        img_left = combined.crop((0, 0, w // 2, h))
        mask_right = combined.crop((w // 2, 0, w, h))
        
        # Redimensionar
        img_tensor = self.transform_img(img_left)
        
        mask_resized = mask_right.resize(self.img_size, Image.NEAREST)
        mask_np = np.array(mask_resized)
        
        # Quantizar máscara RGB em 4 classes semânticas fundamentais
        # Classe 0: Background / Prédios (cinza / azul escuro)
        # Classe 1: Vias / Asfalto (tons de rosa / roxo)
        # Classe 2: Veículos (tons de azul / vermelho vivo)
        # Classe 3: Vegetação (tons predominantemente verdes)
        r, g, b = mask_np[:, :, 0], mask_np[:, :, 1], mask_np[:, :, 2]
        
        mask_labels = np.zeros(self.img_size, dtype=np.int64)
        
        # Vias (Rosa/Roxo: R alto, B alto)
        roads = (r > 100) & (b > 100) & (g < 100)
        # Vegetação (Verde dominante: G > R e G > B)
        vegetation = (g > r) & (g > b) & (g > 80)
        # Veículos (Azul puro: B > 120 e R < 80)
        vehicles = (b > 120) & (r < 80)
        
        mask_labels[roads] = 1
        mask_labels[vehicles] = 2
        mask_labels[vegetation] = 3
        
        mask_tensor = torch.as_tensor(mask_labels, dtype=torch.long)
        
        return img_tensor, mask_tensor

# Datasets e DataLoaders
train_dataset = CityscapesPairsDataset(train_folder, img_size=(128, 128))
val_dataset = CityscapesPairsDataset(val_folder, img_size=(128, 128))

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2 if torch.cuda.is_available() else 0)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=2 if torch.cuda.is_available() else 0)

print(f"✅ Datasets prontos! Treino: {len(train_dataset)} | Validação: {len(val_dataset)}")


## 5. Parte 3 — Implementando a Arquitetura U-Net do Zero em PyTorch

A **U-Net (Ronneberger et al., 2015)** possui uma estrutura perfeitamente simétrica em forma de "U":

```
[Imagem 3x128x128] ──────────────── (Skip Connection 1) ───────────────> [Concat + Conv] ──> [Saída Cx128x128]
        │                                                                        ▲
        ▼ (Down 1: MaxPool)                                                      │ (Up 1: ConvTranspose)
   [64x64x64] ───────────────────── (Skip Connection 2) ───────────────> [Concat + Conv]
        │                                                                        ▲
        ▼ (Down 2: MaxPool)                                                      │ (Up 2: ConvTranspose)
   [128x32x32] ──────────────────── (Skip Connection 3) ───────────────> [Concat + Conv]
        │                                                                        ▲
        ▼ (Down 3: MaxPool)                                                      │ (Up 3: ConvTranspose)
   [256x16x16] ──────────────────── (Skip Connection 4) ───────────────> [Concat + Conv]
        │                                                                        ▲
        ▼ (Down 4: MaxPool)                                                      │ (Up 4: ConvTranspose)
                     [Gargalo / Bottleneck: 512x8x8] ────────────────────────────┘
```

### 🧠 Por que as Skip Connections são Críticas?
- No caminho de descida (*Encoder*), a rede aprende **contexto semântico de alto nível** ("o que está na imagem"), mas perde a localização precisa das bordas devido aos `MaxPool`.
- As **Skip Connections** copiam os mapas de alta resolução do Encoder diretamente para o Decoder, permitindo que a rede reconstrua **bordas nítidas de nível de pixel**!


In [ ]:
class DoubleConv(nn.Module):
    """Bloco Construtivo: [Conv3x3 -> BatchNorm -> ReLU] x 2"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)


class UNet(nn.Module):
    def __init__(self, in_channels=3, num_classes=4):
        super().__init__()
        
        # --- ENCODER (Contração Espacial) ---
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(64, 128))
        self.down2 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(128, 256))
        self.down3 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(256, 512))
        
        # --- BOTTLENECK (Gargalo Mais Profundo) ---
        self.down4 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(512, 512))
        
        # --- DECODER (Expansão e Reconstrução com Skip Connections) ---
        self.up1 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.conv_up1 = DoubleConv(512 + 256, 256)
        
        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.conv_up2 = DoubleConv(256 + 128, 128)
        
        self.up3 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.conv_up3 = DoubleConv(128 + 64, 64)
        
        self.up4 = nn.ConvTranspose2d(64, 64, kernel_size=2, stride=2)
        self.conv_up4 = DoubleConv(64 + 64, 64)
        
        # Camada Final 1x1 que projeta os canais para as num_classes
        self.outc = nn.Conv2d(64, num_classes, kernel_size=1)

    def forward(self, x):
        # Passada pelo Encoder salvando ativações para Skip Connections
        x1 = self.inc(x)         # [B, 64, H, W]
        x2 = self.down1(x1)      # [B, 128, H/2, W/2]
        x3 = self.down2(x2)      # [B, 256, H/4, W/4]
        x4 = self.down3(x3)      # [B, 512, H/8, W/8]
        
        # Bottleneck
        x5 = self.down4(x4)      # [B, 512, H/16, W/16]
        
        # Passada pelo Decoder com Concatenações de Skip Connections
        x = self.up1(x5)
        x = torch.cat([x, x4], dim=1) # Concatenação [256 + 512 = 768 canais]
        x = self.conv_up1(x)
        
        x = self.up2(x)
        x = torch.cat([x, x3], dim=1) # Concatenação [128 + 256 = 384 canais]
        x = self.conv_up2(x)
        
        x = self.up3(x)
        x = torch.cat([x, x2], dim=1) # Concatenação [64 + 128 = 192 canais]
        x = self.conv_up3(x)
        
        x = self.up4(x)
        x = torch.cat([x, x1], dim=1) # Concatenação [64 + 64 = 128 canais]
        x = self.conv_up4(x)
        
        logits = self.outc(x)    # [B, num_classes, H, W]
        return logits

# Instanciar U-Net
unet_model = UNet(in_channels=3, num_classes=4).to(device)
params_count = sum(p.numel() for p in unet_model.parameters())
print(f"🤖 U-Net instanciada com sucesso! Parâmetros Totais: {params_count / 1e6:.2f} M")


## 6. Funções de Perda e Métricas: Cross-Entropy, Dice Loss e mIoU

Em segmentação semântica, o fundo frequentemente ocupa mais de 70% dos pixels. Utilizar apenas Cross-Entropy pode fazer a rede ignorar objetos menores.

Combinamos a **Cross-Entropy por Pixel** com a **Dice Loss (baseada no Coeficiente de Sørensen-Dice)**:

$$\mathcal{L}_{\text{Total}} = \mathcal{L}_{\text{CE}} + \mathcal{L}_{\text{Dice}}$$

$$\mathcal{L}_{\text{Dice}} = 1 - \frac{2 \sum (p_i \cdot y_i) + \epsilon}{\sum p_i + \sum y_i + \epsilon}$$


In [ ]:
class CombinedLoss(nn.Module):
    def __init__(self, num_classes=4, smooth=1.0):
        super().__init__()
        self.num_classes = num_classes
        self.smooth = smooth
        self.ce_loss = nn.CrossEntropyLoss()

    def forward(self, logits, targets):
        # 1. Perda Cross Entropy por Pixel
        ce = self.ce_loss(logits, targets)
        
        # 2. Perda Dice Multiclasse
        probs = F.softmax(logits, dim=1)
        targets_one_hot = F.one_hot(targets, num_classes=self.num_classes).permute(0, 3, 1, 2).float()
        
        intersection = torch.sum(probs * targets_one_hot, dim=(0, 2, 3))
        union = torch.sum(probs, dim=(0, 2, 3)) + torch.sum(targets_one_hot, dim=(0, 2, 3))
        
        dice_score = (2.0 * intersection + self.smooth) / (union + self.smooth)
        dice_loss = 1.0 - torch.mean(dice_score)
        
        return ce + dice_loss

def calculate_iou(pred_mask, target_mask, num_classes=4):
    """Calcula o Mean Intersection over Union (mIoU) sobre as classes presentes"""
    ious = []
    for cls in range(num_classes):
        pred_inds = (pred_mask == cls)
        target_inds = (target_mask == cls)
        
        intersection = (pred_inds & target_inds).sum().item()
        union = (pred_inds | target_inds).sum().item()
        
        if union == 0:
            continue # Ignora classes ausentes no lote
        ious.append(intersection / union)
        
    return np.mean(ious) if len(ious) > 0 else 0.0

print("✅ Funções de Perda Combinada e mIoU definidas com sucesso!")


## 7. Loop de Treinamento da U-Net

Vamos treinar a U-Net por **8 épocas** utilizando o otimizador **AdamW** com agendador de taxa de aprendizado **CosineAnnealingLR**.


In [ ]:
NUM_EPOCHS = 8
LEARNING_RATE = 1e-3

criterion = CombinedLoss(num_classes=4)
optimizer = optim.AdamW(unet_model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

history = {'train_loss': [], 'val_loss': [], 'val_miou': []}

print(f"🚀 Iniciando Treinamento da U-Net por {NUM_EPOCHS} épocas...")
start_time = time.time()

for epoch in range(NUM_EPOCHS):
    unet_model.train()
    running_train_loss = 0.0
    
    pbar = tqdm(train_loader, desc=f"Época {epoch+1}/{NUM_EPOCHS} [Treino]")
    for images, masks in pbar:
        images = images.to(device)
        masks = masks.to(device)
        
        optimizer.zero_grad()
        logits = unet_model(images)
        loss = criterion(logits, masks)
        loss.backward()
        optimizer.step()
        
        running_train_loss += loss.item()
        pbar.set_postfix({'Loss': f"{loss.item():.4f}"})
        
    scheduler.step()
    epoch_train_loss = running_train_loss / len(train_loader)
    
    # Avaliação no conjunto de Validação
    unet_model.eval()
    running_val_loss = 0.0
    val_miou_list = []
    
    with torch.no_grad():
        for images, masks in val_loader:
            images = images.to(device)
            masks = masks.to(device)
            
            logits = unet_model(images)
            loss = criterion(logits, masks)
            running_val_loss += loss.item()
            
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            targets = masks.cpu().numpy()
            
            for p, t in zip(preds, targets):
                val_miou_list.append(calculate_iou(p, t, num_classes=4))
                
    epoch_val_loss = running_val_loss / len(val_loader)
    epoch_val_miou = np.mean(val_miou_list)
    
    history['train_loss'].append(epoch_train_loss)
    history['val_loss'].append(epoch_val_loss)
    history['val_miou'].append(epoch_val_miou)
    
    print(f"✅ Época [{epoch+1:02d}/{NUM_EPOCHS:02d}] — Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Val mIoU: {epoch_val_miou*100:.2f}%")

print(f"\n🎉 Treinamento concluído em {time.time() - start_time:.2f} segundos!")


## 8. Curvas Gráficas de Aprendizado (Perda e mIoU)

Plotamos a convergência das funções de perda e o crescimento da métrica **mIoU**.


In [ ]:
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1: Curvas de Perda
ax1.plot(range(1, NUM_EPOCHS + 1), history['train_loss'], 'o-', label='Perda Treino', color='#0A345D', linewidth=2.5)
ax1.plot(range(1, NUM_EPOCHS + 1), history['val_loss'], 's--', label='Perda Validação', color='#EA580C', linewidth=2.5)
ax1.set_title('Evolução da Função de Perda Combinada (CE + Dice)', fontsize=12, fontweight='bold', color='#0A345D')
ax1.set_xlabel('Época', fontsize=11)
ax1.set_ylabel('Loss', fontsize=11)
ax1.legend(frameon=True, facecolor='white')
ax1.grid(True, alpha=0.3)

# Gráfico 2: Evolução do mIoU
ax2.plot(range(1, NUM_EPOCHS + 1), [m * 100 for m in history['val_miou']], '^-', label='Validação mIoU (%)', color='#16A34A', linewidth=2.5, markersize=8)
ax2.set_title('Evolução do Mean IoU (Jaccard Index) na Validação', fontsize=12, fontweight='bold', color='#0A345D')
ax2.set_xlabel('Época', fontsize=11)
ax2.set_ylabel('mIoU (%)', fontsize=11)
ax2.legend(frameon=True, facecolor='white')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 9. Avaliação Visual Qualitativa no Conjunto de Teste

Vamos comparar lado a lado em 4 colunas:
1. **Imagem Original**
2. **Máscara Real (Ground Truth)**
3. **Máscara Predita pela U-Net**
4. **Alpha-Blend Overlay (Predição sobre a Imagem)**


In [ ]:
CITYSCAPES_COLORS = np.array([
    [50, 50, 50],     # 0: Background / Prédios (Cinza Escuro)
    [236, 72, 153],   # 1: Vias / Asfalto (Rosa Neon)
    [0, 212, 255],    # 2: Veículos (Cyan Infnet)
    [34, 197, 94]     # 3: Vegetação (Verde Vibrante)
], dtype=np.uint8)

def unnormalize(tensor):
    """Desnormaliza tensor [3, H, W] do ImageNet para exibição [H, W, 3]"""
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img_np = tensor.permute(1, 2, 0).cpu().numpy()
    img_np = std * img_np + mean
    return np.clip(img_np, 0.0, 1.0)

unet_model.eval()
with torch.no_grad():
    sample_images, sample_masks = next(iter(val_loader))
    sample_images = sample_images.to(device)
    logits = unet_model(sample_images)
    preds = torch.argmax(logits, dim=1).cpu().numpy()

fig, axes = plt.subplots(3, 4, figsize=(18, 12))

for i in range(3):
    img_display = unnormalize(sample_images[i])
    gt_mask_color = CITYSCAPES_COLORS[sample_masks[i].numpy()]
    pred_mask_color = CITYSCAPES_COLORS[preds[i]]
    
    # Overlay translúcido
    overlay_pred = (img_display * 255 * 0.5 + pred_mask_color * 0.5).astype(np.uint8)
    
    axes[i, 0].imshow(img_display)
    axes[i, 0].set_title(f"Amostra #{i+1} • Foto Original", fontsize=11, fontweight='bold', color='#0A345D')
    axes[i, 0].axis('off')
    
    axes[i, 1].imshow(gt_mask_color)
    axes[i, 1].set_title("Ground Truth (Real)", fontsize=11, fontweight='bold', color='#15803D')
    axes[i, 1].axis('off')
    
    axes[i, 2].imshow(pred_mask_color)
    axes[i, 2].set_title("Predição U-Net", fontsize=11, fontweight='bold', color='#0A345D')
    axes[i, 2].axis('off')
    
    axes[i, 3].imshow(overlay_pred)
    axes[i, 3].set_title("Alpha-Blend Overlay", fontsize=11, fontweight='bold', color='#9333EA')
    axes[i, 3].axis('off')

# Legenda de Classes
legend_elements = [
    patches.Patch(facecolor='#323232', edgecolor='none', label='0: Fundo/Construções'),
    patches.Patch(facecolor='#EC4899', edgecolor='none', label='1: Vias/Asfalto'),
    patches.Patch(facecolor='#00D4FF', edgecolor='none', label='2: Veículos'),
    patches.Patch(facecolor='#22C55E', edgecolor='none', label='3: Vegetação')
]
fig.legend(handles=legend_elements, loc='lower center', ncol=4, fontsize=12, frameon=True, facecolor='white')

plt.suptitle("🎯 Resultados Visuais de Segmentação Semântica com U-Net", fontsize=15, fontweight='bold', color='#0A345D', y=0.99)
plt.tight_layout(rect=[0, 0.05, 1, 0.97])
plt.show()


## 10. 🎓 Desafios Práticos & Exercícios Propostos (Pós-Graduação)

Para consolidar as competências em visão computacional e predição densa:

---

### 🏋️‍♂️ Desafio 1: O Impacto das Skip Connections (Ablation Study)
Crie uma variação da classe `UNet` removendo as concatenações de *Skip Connections* no Decoder (passe apenas o tensor vindo de `ConvTranspose2d`). Treine por 5 épocas e compare a nitidez das bordas das vias e dos veículos.

### 🏋️‍♂️ Desafio 2: Análise de Entropia e Mapa de Incerteza
Calcule a entropia de Shannon de cada pixel $H_{i,j} = -\sum p_c \log(p_c)$ a partir das probabilidades Softmax. Plote o mapa de calor de incerteza e identifique onde o modelo apresenta menor confiança (ex: limites de transição entre asfalto e calçada).

### 🏋️‍♂️ Desafio 3: Substituição do Backbone por uma ResNet Pré-Treinada (Transfer Learning)
Adapte o Encoder da U-Net para utilizar os estágios iniciais de uma `resnet34` pré-treinada no ImageNet (`torchvision.models.resnet34(weights=ResNet34_Weights.DEFAULT)`). Avalie o ganho na velocidade de convergência.
